In [1]:
#Imports
import torch
import random
import os
from numpy import genfromtxt
from lib.tools import random_ranges
import numpy as np
from lib.pinn_auxloss_f import Pinn
import matplotlib.pyplot as plt
from scipy.integrate import odeint
import jax.numpy as jnp
import pandas as pd
import random as rd
from torch import nn
import sys

#Seed specification
random.seed(42)
torch.manual_seed(42)

In [2]:
#Generating a dataset
def acetate_overflow_model(
    t,
    alp=0.8,
    bet=0.4,
    delt=0.2,
    gam=0.6,
):
    def func(y, t):

        X,Y = [y[i] for i in range(len(y))]

        dXdt = X*(alp - bet*Y)

        dYdt = Y*(delt*X - gam)

        return np.array([dXdt, dYdt])

    y0 = [5., 3.] #Conditions initiales
    return odeint(func, y0, t)

t = np.linspace(0,10,30)
y = acetate_overflow_model(np.ravel(t))

##Adding noise in the dataset
def add_random_noise(array: np.ndarray, seed: int = None, noise_scale: float = 0.5) -> np.ndarray:
    if seed is not None:
        np.random.seed(seed)
    noise = np.random.rand(*array.shape)  # random floats in [0, 1)
    signs = np.random.choice([-1, 1], size=array.shape)  # random +/- 1
    return array + signs * noise * noise_scale
data=add_random_noise(y)

#Creating the auxiliary data – for T_init and T_f
data_aux=[torch.tensor([5.,2.]),
              torch.tensor(y[-1])]

In [19]:
data_aux

[tensor([5., 2.]), tensor([3.7627, 3.5012], dtype=torch.float64)]

In [3]:
# Defining the dictionaries

## Dictionary with the true value of parameters
ode_parameters_dict = {"alp":0.8,
                       "bet":0.4,
                       "delt":0.2,
                       "gam":0.6
                       }
## Dictionary with the parameter ranges
ode_parameter_ranges_dict = {"alp":(0.,1.),
                             "bet":(0.,1.),
                             "delt":(0.,1.),
                             "gam":(0.,1.)
                            }
## Dictionary with the std of variables
std_per_variable = np.std(data, axis=0)
variable_standard_deviations_dict = {"X":std_per_variable[0],
                                     "Y":std_per_variable[1]
                                    }

##Dictionary of residuals
ODE_residuals = {"ode_1" : 
                 lambda var_dict,d_dt_var_dict,value,min_var_dict,max_var_dict : 
                     d_dt_var_dict["X"] - var_dict["X"]*(value["alp"] - value["bet"]*var_dict["Y"]),
                 "ode_2" : 
                 lambda var_dict,d_dt_var_dict,value,min_var_dict,max_var_dict : 
                     d_dt_var_dict["Y"] - var_dict["Y"]*(value["delt"]*var_dict["X"] - value["gam"])
                }

In [4]:
##Names of the variables associated with data
observables = ["X","Y"] 

##Dictionaries for the data and no-data variables (and associated data)
variable_data = {"X": data[:,0], "Y": data[:,1]} 
variable_no_data  = {}

##Time points specification
data_t = t

In [6]:
#List of the names of unknown parameters
parameter_names = ["alp",
                   "bet",
                   "delt",
                   "gam"]

#Specifying the ranges from the dictionary
ranges = random_ranges([ode_parameters_dict[key] for key in parameter_names],scale=20)
for i,name in enumerate(parameter_names):
    if name in ode_parameter_ranges_dict:
        ranges[i]= ode_parameter_ranges_dict[name]
        
#Specifying the constants, i.e. the true values of parameters
constants_dict = ode_parameters_dict

In [7]:
# Training parameters
epoch_number = 150000

# Optimizer parameters
optimizer_type = "Adam"
optimizer_hyperparameters = {"lr":1e-4, "betas":(0.9, 0.8)}

# Scheduler parameters
scheduler_hyperparameters = {"base_lr":1e-4,
                             "max_lr":1e-4,
                             "step_size_up":100,
                             "scale_mode":"exp_range",
                             "gamma":0.999,
                             "cycle_momentum":False}

#Specifying the guiding weighting of the PINN (i.e. the weights that will multiply the balancing method output)
residual_weights=[1,1]

# Loss balancing method
multiple_loss_method = "prior_losses"

In [8]:
#Creating PINN
pinn_cell = Pinn(ode_residual_dict=ODE_residuals,
                 ranges=ranges,
                 data_t=data_t,
                 variables_data=variable_data,
                 variables_no_data=variable_no_data,
                 data_aux= data_aux, # [danilo]{def: data_aux_1mM},
                 parameter_names=parameter_names,
                 optimizer_type=optimizer_type,
                 optimizer_hyperparameters=optimizer_hyperparameters,
                 scheduler_hyperparameters=scheduler_hyperparameters,
                 constants_dict=constants_dict,
                 
                 #Loss balancing specification
                 multi_loss_method=multiple_loss_method,
                 residual_weights=residual_weights,
                 variable_fit_weights=None,
                 auxiliary_fit_weights=None,
                 
                 #SoftAdapt parameters
                 soft_adapt_beta=0.1,
                 soft_adapt_t=1,
                 soft_adapt_normalize=True,
                 soft_adapt_by_type=True,
                 soft_adapt_eps=10E-8,
                 soft_adapt_warming=-1,
                 
                 #Increments parametrisation
                 incr_residual_weight=20000,
                 increment=1E2,
                 
                 #Prior_losses parameters
                 prior_losses_t=100,
                 
                 #Wang parameters
                 wang_residual = True,
                 wang_t=1,
                 wang_alpha=0.9,
                 wang_epsilon=1E-8,
                 wang_warming=-1,
                 
                 #NN parameters
                 net_hidden=7,
                 activation_function=nn.Softplus(),
                 optuna=False)

In [16]:
ODE_residuals["ode_1"]()

<function __main__.<lambda>(var_dict, d_dt_var_dict, value, min_var_dict, max_var_dict)>